This notebook computes the geodesic distance between pairs of grocery stores.

In [1]:
# import sys
# !{sys.executable} -m pip install geopy

In [2]:
## Modules

import geopandas as gpd
import pandas as pd
import numpy as np

import json
import urllib.request
import geopandas as gpd
import pandas as pd
import numpy as np
import geopy.distance
import osmnx as ox
from shapely.geometry import Point, LineString, Polygon
from shapely.geometry import MultiPoint

In [3]:
# Location of data
datadir = '../Data/'

# On AKB's computer while testing
#datadir = '../../tda-resources-dallas/Dallas/'

In [8]:
#geodesic distance definitions 
#

def get_dist(a, b):
    origin = str(a.y)+','+str(a.x)
    destination = str(b.y)+','+str(b.x)
    dist = geopy.distance.geodesic(origin, destination)
    return dist.meters
    
#def get_walk_time(a,b):
#    dist = get_dist(a,b)
#    return dist

def calc_twalk_matrix(locs):
                      #neighbours):
    """
    Input:
    polls is a geodataframe storing locations as Points
    neighbours is a 0/1 array representing which locations we would like to compute distances between
    
    Returns: A matrix of distances between locations
    """
    print(len(locs))
    N = locs.shape[0]
    print(locs.shape)
    t_walk = np.zeros((N,N))
    print(t_walk.shape)
    
    for index_i, location_a in locs.iterrows():
        for index_j, location_b in locs.iterrows():
            #print(index_i, index_j)
            if index_i != index_j:
                t_walk[index_i,index_j] = get_dist(location_a.geometry,location_b.geometry)
    return t_walk

In [5]:
groc_sites = gpd.read_file('../Data/geo_export_872fcb6c-fbde-4264-ae77-8858a604ed0e.shp' ) #import grocery store locations

In [6]:
#Loading street network

# Our set of grocery stores has points in Collin County 
# (It is City of Dallas, which actually crosses the county line (while also not including non-COD
#     places in Dallas Co)
# This can cause errors when attempting to find path between nodes. 

use_bbox_for_grid = False

if use_bbox_for_grid:
    # To include the southern piece of Collin Co, use bbox
    # bbox = [-97.1,32.5,-96.45,33.1]
    # bbox = [-97.8080, 32.7725, -96.7900, 32.7915]
    coords_list = [-96.808, 32.780, -96.788, 32.807]
    G = ox.graph_from_bbox(coords_list,network_type='walk', simplify=False)
   # G_df = ox.geocode.geocode_to_(bbox,network_type='walk', simplify=False)
else: 
    place = 'Dallas, Texas, United States' #!! replace with desired city here
    #G = ox.graph_from_place(place,network_type='walk', buffer_dist = 5000, simplify=False)
    G = ox.graph_from_place(place,network_type='walk', simplify=False)

# coords = [
    
#     (-95.8080, 32.7725),  # Point 1
#       # Point 2
#     (-96.7900, 32.7725),
#       # Point 3
#     (-96.7900, 32.7915),
#     (-96.8080, 32.7915)
#    # Point 4
# ]
# four_tuples = [
#     (-96.808, 32.780),
#     (-96.808, 32.807),
#     (-96.788, 32.807),# Top-Left (Northwest)
#     (-96.788, 32.780) # Top-Right (Northeast)
#      # Bottom-Right (Southeast)
#       # Bottom-Left (Southwest)
# ]

# # Create the bounding area polygon
# geo_area = Polygon(four_tuples)

# geo_area
# # print(f"Is this area a valid shape? {geo_area.is_valid}")

# # G.draw()

# groc_sites = gpd.read_file(datadir+'geo_export_872fcb6c-fbde-4264-ae77-8858a604ed0e.shp') # load grocery store sites

# # groc_sites.head()
# groc_sites_select = groc_sites[groc_sites['city'] == 'Dallas']
# groc_sites_select.head()

# groc_sites_select['intersects'] = groc_sites_select.within(geo_area)
# groc_sites_select_inner = groc_sites_select[groc_sites_select['intersects'] == True]
# groc_sites_select_inner = groc_sites_select_inner.reset_index()

# groc_sites_select_inner = groc_sites_select_inner.drop('index', axis = 1)
#     # head()

In [14]:
# groc_sites_select_inner.head()

,address,city,status,store_name,geometry,intersects
0,3524 McKinney Ave,Dallas,Open,Albertsons,POINT (-96.79712 32.80557),True
1,2305 N Central Expy,Dallas,Open,Walmart Market,POINT (-96.79371 32.80061),True
2,2417 N Haskell Rd,Dallas,Open,Target,POINT (-96.79175 32.80481),True
3,4241 Capitol Ave,Dallas,Open,Kroger,POINT (-96.78975 32.80657),True
4,McKinney Ave at Routh St,Dallas,Opening,Whole Foods Market,POINT (-96.80161 32.79554),True


In [9]:
# Geodesic distance
dal_geodesic_distance_meters = calc_twalk_matrix(groc_sites)
                                                 #_select_inner)
                                                 #, groc_neighbours)

138
(138, 5)
(138, 138)


In [10]:
# Change units to seconds by using an average walk speed
walk_speed = 1.42 # meters/sec

In [11]:
dal_geodesic_distance_seconds = dal_geodesic_distance_meters/walk_speed #compute "walk time" (in seconds) for geodesic distance

In [12]:
dal_geodesic_distance_seconds

array([[    0.        ,  9232.54821472, 12308.32868523, ...,
         4727.25316419, 12334.67860647, 14379.47238599],
       [ 9232.54821472,     0.        ,  7403.76409149, ...,
         9771.58343874,  4288.53632159,  8891.49369026],
       [12308.32868523,  7403.76409149,     0.        , ...,
         9583.68358522,  4465.12005008,  2071.44151773],
       ...,
       [ 4727.25316419,  9771.58343874,  9583.68358522, ...,
            0.        , 11278.61443008, 11557.63258959],
       [12334.67860647,  4288.53632159,  4465.12005008, ...,
        11278.61443008,     0.        ,  5209.59900847],
       [14379.47238599,  8891.49369026,  2071.44151773, ...,
        11557.63258959,  5209.59900847,     0.        ]], shape=(138, 138))

In [13]:
np.savez('dal_geodesic_distance_data', dal_geodesic_distance_meters = dal_geodesic_distance_meters,
         walk_speed = walk_speed, dal_geodesic_distance_seconds = dal_geodesic_distance_seconds)